# §4.4 — Advanced Spatial Analyses

Mark Correlation Function (MCF), Composite Severity Index (CSI), and DBSCAN on morphological features.

**Inputs:** `data/clusters_typed.csv`  
**Outputs:** Figures 9, 10, 11

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN as skDBSCAN

from metrics.spatial import mark_correlation_function
from severity import composite_severity_index

In [ ]:
df = pd.read_csv('../data/clusters_typed.csv').dropna(subset=['cx', 'cy', 'fractal_dim', 'clark_evans'])
coords = df[['cx', 'cy']].values
print(f'{len(df):,} clusters with complete metrics')

## Figure 9 — Mark Correlation Function (MCF)

Mark = fractal dimension per cluster centroid. k_m(r) > 1 means nearby clusters share similar complexity.

In [ ]:
r_values = np.linspace(10, 500, 50)
marks = df['fractal_dim'].values
km = mark_correlation_function(coords, marks, r_values, dr=10.0)

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
ax.plot(r_values, km, color='#0072B2', lw=2, label='k_m(r)')
ax.axhline(1.0, color='#D55E00', lw=2, ls='--', label='CSR baseline')
ax.set_xlabel('Distance r (m)', fontsize=12)
ax.set_ylabel('k_m(r)', fontsize=12)
ax.set_title('Mark Correlation Function — fractal dimension', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../figures/fig9_mcf.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 10 — Composite Severity Index (CSI)

CSI = fractal_dim × (1 / clark_evans) × n_points

In [ ]:
df['csi'] = composite_severity_index(
    df['fractal_dim'].values,
    df['clark_evans'].values,
    df['n_points'].values
)

print(df['csi'].describe())
print(f"Clusters with CSI > 10: {(df['csi'] > 10).sum():,}")
print(f"Clusters with CSI > 15: {(df['csi'] > 15).sum():,}")

fig, ax = plt.subplots(figsize=(10, 9), dpi=150)
sc = ax.scatter(df['cx'], df['cy'], c=df['csi'],
                cmap='YlOrRd', s=5, alpha=0.7, vmax=df['csi'].quantile(0.99))
plt.colorbar(sc, ax=ax, label='CSI')
ax.set_xlabel('Easting (m)', fontsize=11)
ax.set_ylabel('Northing (m)', fontsize=11)
ax.set_title('Composite Severity Index (CSI) spatial distribution', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/fig10_csi.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 11 — DBSCAN on morphological feature space

Groups clusters with similar morphology (eps=0.3 in standardised feature space).

In [ ]:
MORPH_FEATURES = ['fractal_dim', 'clark_evans', 'aspect_ratio']
data = df[MORPH_FEATURES].dropna()
data_scaled = StandardScaler().fit_transform(data)

db = skDBSCAN(eps=0.3, min_samples=3).fit(data_scaled)
labels = db.labels_
n_groups = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()
print(f'Morphological groups: {n_groups:,}  (expect ~191)')
print(f'Noise points: {n_noise:,}  (expect ~3,392)')

plot_df = df.loc[data.index].copy()
plot_df['morph_group'] = labels

from scipy.stats import gaussian_kde
fig, ax = plt.subplots(figsize=(10, 9), dpi=150)

# KDE contour background
kde = gaussian_kde(coords.T, bw_method='scott')
xi = np.linspace(coords[:, 0].min(), coords[:, 0].max(), 150)
yi = np.linspace(coords[:, 1].min(), coords[:, 1].max(), 150)
Xi, Yi = np.meshgrid(xi, yi)
Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
ax.contour(Xi, Yi, Zi, levels=8, colors='grey', alpha=0.4, linewidths=0.5)

# Noise points
noise = plot_df[plot_df.morph_group == -1]
ax.scatter(noise['cx'], noise['cy'], s=3, color='lightgrey', alpha=0.4, label='Noise')

# Grouped clusters (colour by group, up to 20 colours)
non_noise = plot_df[plot_df.morph_group != -1]
cmap = plt.get_cmap('tab20')
for g, grp in non_noise.groupby('morph_group'):
    ax.scatter(grp['cx'], grp['cy'], s=8, color=cmap(g % 20), alpha=0.7)

ax.set_xlabel('Easting (m)', fontsize=11)
ax.set_ylabel('Northing (m)', fontsize=11)
ax.set_title(f'DBSCAN on morphological features — {n_groups} groups identified', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../figures/fig11_morph_dbscan.png', dpi=300, bbox_inches='tight')
plt.show()